# ModelGuard Experiment Notebook


### Imports

In [ ]:
from pathlib import Path

from flowguard.experiments.catalog import (
    SUPPORTED_ATTACKS,
    SUPPORTED_MODES,
    SUPPORTED_PREDICTION_DEFENSES,
)
from flowguard.experiments.factory import build_experiment_matrix, build_experiment_spec
from flowguard.experiments.spec import AttackKind
from flowguard.orchestration import run_experiment, run_experiment_suite
from flowguard.visualization import (
    build_attack_query_animation,
    load_attack_visualization_run,
    plot_attack_queries_projection,
)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RUNS_DIR = ROOT / "runs" / "notebook"
RUNS_DIR.mkdir(parents=True, exist_ok=True)


### Available Attacks, Modes and Defenses

In [ ]:
MODE_TABLE = [(entry.attack_kind.value, entry.mode_name, entry.description) for entry in SUPPORTED_MODES]
DEFENSE_TABLE = list(SUPPORTED_PREDICTION_DEFENSES)
MODE_TABLE, DEFENSE_TABLE


### Base Config

In [ ]:
# Passe diese Basiskonfiguration an dein Setup an.
BASE_CONFIG = {
    "dataset": "CIFAR10",
    "target_architecture": "vgg16_bn",
    "substitute_architecture": "vgg16_bn",
    "query_dataset": "CIFAR10",
    "query_budget": 10000,
    "query_transfer_set_size": 2000,
    "verbose": True,
    "target_checkpoint_dir": str(
        RUNS_DIR / "training-victim-cifar10-vgg16_bn-nodefense" / "target_model"
    ),
    "target_device": "cuda",
    "substitute_device": "cuda",
    "distributed": True,
    "num_clients": 100,  
    "attack_batch_size": 8,
    "training_batch_size": 128,
    "epochs": 2,
}

single_spec = build_experiment_spec(
    name="single-prada-none",
    attack_kind=AttackKind.PRADA,
    attack_mode="prada",
    prediction_defense="none",
    **BASE_CONFIG,
)
#single_spec.training.epochs = 4
#single_spec.training.lr = 0.005
#single_spec.attack.seed_size = 800
#single_spec.attack.epsilon = 0.08
#single_spec.attack.steps = 10
single_spec.attack.extra.update(
    {
        "duplication_rounds": 2,
 #       "expansion_factor": 2,
 #       "step_size": 0.008,
#        "max_iter": 10,
 #       "decay": 1.0,
  #      "hyperparameter_search_budget": 0,
    }
)

single_spec


In [ ]:
# Einzelnes Experiment starten.
# Achtung: Für echte Läufe Budget/Epochs hochsetzen und ggf. Checkpoint-Pfade angeben.
single_result = run_experiment(single_spec, output_dir=RUNS_DIR / single_spec.name)
single_result


### MAZE

In [ ]:
maze_config = {
    **BASE_CONFIG,
    "query_dataset": "TinyImageNet200",
    "query_transfer_set_size": None,
}

maze_spec = build_experiment_spec(
    name="single-maze-none",
    attack_kind=AttackKind.MAZE,
    attack_mode="maze",
    prediction_defense="none",
    attack_extra={
        "latent_dim": 100,
        "iter_gen": 1,
        "iter_clone": 5,
        "iter_exp": 1,
        "ndirs": 4,
        "opt": "adam",
        "lr_clone": 0.01,
        "lr_gen": 1e-4,
        "lr_dis": 1e-4,
        "alpha_gan": 0.0,
        "num_seed": 0,
        "lambda1": 10.0,
        "finite_difference_epsilon": 1e-3,
        "log_iter": 1000,
        "disable_pbar": False,
    },
    **maze_config,
)

maze_result = run_experiment(maze_spec, output_dir=RUNS_DIR / maze_spec.name)
maze_result

### DisGUIDE

In [ ]:
disguide_config = {
    **BASE_CONFIG,
    "query_dataset": "TinyImageNet200",
    "query_transfer_set_size": None,
}

disguide_spec = build_experiment_spec(
    name="single-disguide-none",
    attack_kind=AttackKind.DISGUIDE,
    attack_mode="disguide",
    prediction_defense="none",
    attack_extra={
        "ensemble_size": 2,
        "latent_dim": 256,
        "g_iter": 1,
        "d_iter": 1,
        "rep_iter": 3,
        "replay_size": 100000,
        "lambda_div": 0.2,
        "loss": "kl",
        "epoch_itrs": 10,
        "scheduler": "multistep",
        "steps": [0.4, 0.8],
        "scale": 0.3,
        "lr_student": 0.01,
        "lr_generator": 1e-4,
        "disable_pbar": False,
    },
    **disguide_config,
)

disguide_result = run_experiment(disguide_spec, output_dir=RUNS_DIR / disguide_spec.name)
disguide_result

### Attack Visualizations


In [ ]:
single_viz = load_attack_visualization_run(disguide_spec, disguide_result)

projection_figure = plot_attack_queries_projection(
    single_viz,
    projection_method="pca", # pca, tsne, umap
    projection_dims=2,
)

### Attack Query Animation

In [ ]:
animation_figure = build_attack_query_animation(
    single_viz,
    projection_method="pca",
    projection_dims=2,
)
animation_figure

### Experiment using many attacks

In [ ]:
# Beispiel: kleine Suite zum Validieren der Infrastruktur.
suite_specs = build_experiment_matrix(
    base_name="suite",
    dataset=BASE_CONFIG["dataset"],
    target_architecture=BASE_CONFIG["target_architecture"],
    substitute_architecture=BASE_CONFIG["substitute_architecture"],
    query_dataset=BASE_CONFIG["query_dataset"],
    query_budget=BASE_CONFIG["query_budget"],
    target_checkpoint_dir=BASE_CONFIG["target_checkpoint_dir"],
    attack_batch_size=BASE_CONFIG["attack_batch_size"],
    training_batch_size=BASE_CONFIG["training_batch_size"],
    epochs=BASE_CONFIG["epochs"],
    target_device=BASE_CONFIG["target_device"],
    substitute_device=BASE_CONFIG["substitute_device"],
    distributed=False,
    num_workers=1,
    attack_modes=["naive", "jbtr"],
    defenses=["none", "reverse_sigmoid"],
)
len(suite_specs), [spec.name for spec in suite_specs]


In [ ]:
# Mehrere Experimente als Suite ausführen.
suite_results = run_experiment_suite(suite_specs, output_root=RUNS_DIR / "suite")
[(result.experiment_name, result.evaluation_summary.metrics) for result in suite_results]


## Defenses

In [ ]:
import logging
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Global Query Budget für alle Angriffe
GLOBAL_QUERY_BUDGET = 200
logger.info(f"Using a global query budget of {GLOBAL_QUERY_BUDGET} for all defenses")


### Prediction Defenses

In [ ]:
prediction_results = defaultdict(dict)

for attack_def in SUPPORTED_MODES:
    attack_kind = attack_def.attack_kind
    attack_mode = attack_def.mode_name
    logger.info(f"--- Evaluating attack: {attack_kind.value}/{attack_mode} ---")
    
    # Base kwargs for the attack
    attack_kwargs = {**BASE_CONFIG, "query_budget": GLOBAL_QUERY_BUDGET}
    
    # Attack-specific extras (similar to the earlier cells)
    if attack_kind == AttackKind.MAZE:
        attack_kwargs.update({
            "query_dataset": "TinyImageNet200",
            "query_transfer_set_size": None,
            "attack_extra": {
                "latent_dim": 100, "iter_gen": 1, "iter_clone": 5, "iter_exp": 1, "ndirs": 4, 
                "opt": "adam", "lr_clone": 0.01, "lr_gen": 1e-4, "lr_dis": 1e-4, "alpha_gan": 0.0, 
                "num_seed": 0, "lambda1": 10.0, "finite_difference_epsilon": 1e-3, 
                "log_iter": 1000, "disable_pbar": False,
            }
        })
    elif attack_kind == AttackKind.DISGUIDE:
        attack_kwargs.update({
            "query_dataset": "TinyImageNet200",
            "query_transfer_set_size": None,
            "attack_extra": {
                "ensemble_size": 2, "latent_dim": 256, "g_iter": 1, "d_iter": 1, "rep_iter": 3, 
                "replay_size": 100000, "lambda_div": 0.2, "loss": "kl", "epoch_itrs": 10,
                "scheduler": "multistep", "steps": [0.4, 0.8], "scale": 0.3, 
                "lr_student": 0.01, "lr_generator": 1e-4, "disable_pbar": False,
            }
        })
    elif attack_kind == AttackKind.PRADA:
        attack_kwargs.update({"attack_extra": {"duplication_rounds": 2}})
    
    # Run against all prediction defenses
    for defense in SUPPORTED_PREDICTION_DEFENSES:
        experiment_name = f"eval-pred-{attack_mode}-{defense}"
        logger.info(f"  -> Testing Prediction Defense: {defense}")
        
        try:
            spec = build_experiment_spec(
                name=experiment_name,
                attack_kind=attack_kind,
                attack_mode=attack_mode,
                prediction_defense=defense,
                **attack_kwargs
            )
            result = run_experiment(spec, output_dir=RUNS_DIR / experiment_name)
            metrics = result.evaluation_summary.metrics
            prediction_results[attack_mode][defense] = metrics
            logger.info(f"     Success: Acc={metrics.get('accuracy', 0):.2f}%, "
                        f"Fid={metrics.get('fidelity', 0):.2f}%, "
                        f"Joint={metrics.get('joint_accuracy', 0):.2f}%")
        except Exception as e:
            logger.error(f"     Failed: {e}")
            prediction_results[attack_mode][defense] = None


In [ ]:
def plot_prediction_defense_results(results_matrix):
    attack_modes = list(results_matrix.keys())
    defenses = list(SUPPORTED_PREDICTION_DEFENSES)

    metrics_to_plot = ["accuracy", "fidelity", "joint_accuracy"]
    titles = ["Substitute Accuracy", "Substitute Fidelity", "Joint Accuracy"]
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    x = np.arange(len(defenses))
    width = 0.8 / len(attack_modes) if attack_modes else 0.8
    
    for metric_idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
        ax = axes[metric_idx]
        
        for i, mode in enumerate(attack_modes):
            values = []
            for d in defenses:
                res = results_matrix[mode].get(d)
                values.append(res[metric] if res else 0)
            
            # Offset bars per attack mode
            pos = x + (i - len(attack_modes)/2 + 0.5) * width
            ax.bar(pos, values, width, label=mode)
            
        ax.set_title(title, fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(defenses, rotation=45, ha="right", fontsize=12)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Percentage %")
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        
        if metric_idx == 1:
            ax.legend(title="Attack Mode", bbox_to_anchor=(0.5, -0.3), loc="upper center", ncol=4)
            
    plt.tight_layout()
    plt.show()

if prediction_results:
    plot_prediction_defense_results(prediction_results)


### Query Defenses

In [ ]:
SUPPORTED_MODES

In [ ]:
query_defense_results = defaultdict(dict)
query_defenses_to_test = ["flow_matching", "none"] 
fm_checkpoint = str(ROOT / "runs/flow_matching/cifar10_notebook/checkpoint_latest.pt")

# We want to know how many normal baseline queries are falsely blocked
baseline_qd_params = {
    "fm_checkpoint_path": fm_checkpoint, 
    "likelihood_threshold": -3050.0, 
    "audit_only": True
}
logger.info("--- Evaluating BASELINE (Normal Validation Data) with Flow Matching ---")
baseline_spec = build_experiment_spec(
    name="eval-query-baseline-flow_matching",
    attack_kind=AttackKind.TRANSFER_SET, # Baseline dummy
    attack_mode="naive",
    query_defense="flow_matching",
    query_defense_parameters=baseline_qd_params,
    **{**BASE_CONFIG, "query_budget": GLOBAL_QUERY_BUDGET}
)
try:
    baseline_result = run_experiment(baseline_spec, output_dir=RUNS_DIR / baseline_spec.name)
    blocks = [r.metadata.get("flow_matching_blocked", [False]) for r in baseline_result.query_summary.history.records]
    total_blocked_baseline = sum([sum(b) for b in blocks])
    logger.info(f"     Baseline Queries Blocked (False Positives): {total_blocked_baseline} / {GLOBAL_QUERY_BUDGET}")
    query_defense_results["baseline"]["flow_matching"] = {
        "queries": GLOBAL_QUERY_BUDGET, "blocked_count": total_blocked_baseline, "metrics": baseline_result.evaluation_summary.metrics
    }
except Exception as e:
    logger.error(f"     Baseline evaluation failed: {e}")

for attack_def in SUPPORTED_MODES[-4:]:
    attack_kind = attack_def.attack_kind
    attack_mode = attack_def.mode_name
    logger.info(f"--- Evaluating attack: {attack_kind.value}/{attack_mode} ---")
    
    attack_kwargs = {**BASE_CONFIG, "query_budget": GLOBAL_QUERY_BUDGET}
    
    # Gleiche Attack Extras wie oben
    if attack_kind == AttackKind.MAZE:
        attack_kwargs.update({
            "query_dataset": "TinyImageNet200", "query_transfer_set_size": None,
            "attack_extra": {"latent_dim": 100, "iter_gen": 1, "iter_clone": 5, "iter_exp": 1, "ndirs": 4, 
                             "opt": "adam", "lr_clone": 0.01, "lr_gen": 1e-4, "lr_dis": 1e-4, "alpha_gan": 0.0, 
                             "num_seed": 0, "lambda1": 10.0, "finite_difference_epsilon": 1e-3, 
                             "log_iter": 1000, "disable_pbar": False}
        })
    elif attack_kind == AttackKind.DISGUIDE:
        attack_kwargs.update({
            "query_dataset": "TinyImageNet200", "query_transfer_set_size": None,
            "attack_extra": {"ensemble_size": 2, "latent_dim": 256, "g_iter": 1, "d_iter": 1, "rep_iter": 3, 
                             "replay_size": 100000, "lambda_div": 0.2, "loss": "kl", "epoch_itrs": 10,
                             "scheduler": "multistep", "steps": [0.4, 0.8], "scale": 0.3, 
                             "lr_student": 0.01, "lr_generator": 1e-4, "disable_pbar": False}
        })
    elif attack_kind == AttackKind.PRADA:
        attack_kwargs.update({"attack_extra": {"duplication_rounds": 2}})

    for qd in query_defenses_to_test:
        experiment_name = f"eval-query-{attack_mode}-{qd}"
        logger.info(f"  -> Testing Query Defense: {qd}")
        
        qd_params = {}
        if qd == "flow_matching":
            qd_params = {
                "fm_checkpoint_path": fm_checkpoint,
                "likelihood_threshold": -3050.0,   # Setzen den im Test ermittelten Threshold
                "audit_only": True                 # Abstürze verhindern, wir loggen nur ob geblockt
            }
            
        try:
            spec = build_experiment_spec(
                name=experiment_name,
                attack_kind=attack_kind,
                attack_mode=attack_mode,
                query_defense=qd,
                query_defense_parameters=qd_params,
                **attack_kwargs
            )
            
            result = run_experiment(spec, output_dir=RUNS_DIR / experiment_name)
            queries_used = result.query_summary.total_queries
            metrics = result.evaluation_summary.metrics
            
            # Auslesen der Blocked-Statistiken aus der Engine-History 
            # audit_only = True sichert, dass der Ansatz auf die Engine und den QueryHistory Log zugeht
            blocked_count = 0
            if result.query_summary.history:
                blocks = [r.metadata.get("queries_blocked", [False]) for r in result.query_summary.history.records]
                blocked_count = sum([sum(b) for b in blocks])
            
            query_defense_results[attack_mode][qd] = {
                "blocked": False, # Technical block (abort execution)
                "blocked_count": blocked_count, 
                "queries": queries_used, 
                "metrics": metrics
            }
            logger.info(f"     Finished. Queries: {queries_used}, Blocked/Flagged: {blocked_count}, Acc: {metrics.get('accuracy', 0):.2f}%")
            
        except Exception as e:
            logger.error(f"     Failed to run: {e}")
            query_defense_results[attack_mode][qd] = {"blocked": False, "error": str(e), "failed": True}

In [ ]:
def plot_query_defense_results(results_matrix):
    attack_modes = list([k for k in results_matrix.keys() if k != "baseline"])
    defenses = ["none", "flow_matching"]
    
    # 1. Bar chart of block-rates (Detection Rates) for flow matching
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    x = np.arange(len(attack_modes))
    width = 0.35
    
    # Left Plot: Blocked queries by Flow Matching
    ax1 = axes[0]
    fm_blocks = []
    for mode in attack_modes:
        res = results_matrix[mode].get("flow_matching", {})
        fm_blocks.append(res.get("blocked_count", 0) / max(1, res.get("queries", GLOBAL_QUERY_BUDGET)) * 100)
    
    baseline_res = results_matrix.get("baseline", {}).get("flow_matching", {})
    baseline_block_ratio = (baseline_res.get("blocked_count", 0) / max(1, baseline_res.get("queries", GLOBAL_QUERY_BUDGET))) * 100
    
    bars = ax1.bar(x, fm_blocks, width, color='purple')
    ax1.axhline(baseline_block_ratio, color='red', linestyle='--', label=f'Baseline False Positives ({baseline_block_ratio:.1f}%)')
    ax1.set_title("Flow Matching Detection Rate (Threshold: -3050)")
    ax1.set_xticks(x)
    ax1.set_xticklabels([m.upper() for m in attack_modes], rotation=45, ha="right")
    ax1.set_ylim(0, 105)
    ax1.set_ylabel("Blocked Queries (%)")
    ax1.legend()
    
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2, f"{bar.get_height():.1f}%", ha='center', va='bottom')

    # Right Plot: Resulting Model Accuracy
    ax2 = axes[1]
    acc_none = []
    acc_fm = []
    
    for mode in attack_modes:
        res_none = results_matrix[mode].get("none", {})
        acc_none.append(res_none.get("metrics", {}).get("accuracy", 0) if not res_none.get("failed") else 0)
        
        res_fm = results_matrix[mode].get("flow_matching", {})
        acc_fm.append(res_fm.get("metrics", {}).get("accuracy", 0) if not res_fm.get("failed") else 0)
        
    pos = np.arange(len(attack_modes))
    ax2.bar(pos - width/2, acc_none, width, label='No Defense', color='blue')
    ax2.bar(pos + width/2, acc_fm, width, label='Flow Matching', color='orange')
    
    ax2.set_title("Resulting Substitute Accuracy")
    ax2.set_xticks(pos)
    ax2.set_xticklabels([m.upper() for m in attack_modes], rotation=45, ha="right")
    ax2.set_ylim(0, 105)
    ax2.set_ylabel("Accuracy (%)")
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

if query_defense_results:
    plot_query_defense_results(query_defense_results)

### Visualize Flow Matching Decision Boundaries
Die Flow Matching Query Defense schätzt die "Log-Likelihood" für eingehende Bilder ab. Generierte Angriffs-Querries (insb. synthethische wie MAZE oder DisGUIDE) formen eigene Verteilungen abseits der regulären Testdaten. Hier visualisieren wir diese Scoring-Werte, um einen passenden Schwellenwert (`likelihood_threshold`) ableiten zu können.

In [ ]:
import torch
import torchvision.datasets as dset
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from flowguard.defenses.query.flow_matching import FlowMatchingQueryDefense
from tqdm.notebook import tqdm 

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Flow Matching Defense Instanz manuell laden 
# Wir erhöhen die Tolleranzen (atol/rtol) und drehen die step_size hoch, was extrem viel Rechenzeit spart!
print(f"Loading Flow Matching checkpoint from: {ROOT / 'runs/flow_matching/cifar10_notebook/checkpoint_latest.pt'}")
fm_checkpoint = str(ROOT / "runs/flow_matching/cifar10_notebook/checkpoint_latest.pt")
defense = FlowMatchingQueryDefense(
    fm_checkpoint_path=fm_checkpoint, 
    dataset_name='CIFAR10', 
    device=device,
    inputs_normalized=True,
    step_size=0.1,  # Voreinstellung ist oft 0.05, 0.1 halbiert die Laufzeit
    atol=1e-3,      # Voreinstellung ist oft 1e-5
    rtol=1e-3
)

# Hilfsfunktion für Batched-Likelihood Berechnung
def batched_log_likelihood(defense_instance, tensor_imgs, batch_size=32):
    lls = []
    # Schneide in kleine Häppchen
    for i in range(0, len(tensor_imgs), batch_size):
        batch = tensor_imgs[i:i+batch_size].to(device)
        with torch.no_grad():
            ll = defense_instance._estimate_log_likelihood(batch)
            lls.append(ll.cpu())
    return torch.cat(lls).numpy()

# 2. Likelihood von regulären Validierungs-Bildern erzeugen (Baseline)
print("Estimating log-likelihood for normal CIFAR-10 validation images...") 
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Korrekter Pfad des Frameworks (data/cifar10)
print(f"Loading CIFAR-10 validation images from: {ROOT / 'data' / 'cifar10'}")
ds = dset.CIFAR10(root=str(ROOT / "data" / "cifar10"), train=False, download=False, transform=transform)
normal_imgs = torch.stack([ds[i][0] for i in range(128)])

print("Estimating log-likelihoods for normal images (Batched)...")
ll_normal = batched_log_likelihood(defense, normal_imgs, batch_size=32)

# 3. Likelihood der Attack-Samples aus den Transfersets extrahieren
print("Estimating log-likelihood for attack samples...")
attack_lls = {}
attack_list = ["jbtr", "prada", "maze", "disguide"]

for attack in tqdm(attack_list, desc="Processing attacks"):
    path = ROOT / f"runs/notebook/eval-query-{attack}-flow_matching/attack/transferset.pickle"
    if path.exists():
        samples = torch.load(path, weights_only=False)
        atk_tensors = []
        for i in range(min(128, len(samples))):
            x = samples[i][0]
            atk_tensors.append(x.clone().detach() if isinstance(x, torch.Tensor) else torch.tensor(x))
            
        imgs_atk = torch.stack(atk_tensors)
        
        # Falls Upsampling erfolgte (z.B. bei MAZE/DisGUIDE mit TinyImageNet)
        if imgs_atk.shape[-1] != 32:
            imgs_atk = torch.nn.functional.interpolate(imgs_atk, size=(32, 32))
        
        attack_lls[attack] = batched_log_likelihood(defense, imgs_atk, batch_size=32)

# 4. Verteilungen aufzeichnen
plt.figure(figsize=(14, 7))

# Normal Distribution
plt.hist(ll_normal, bins=25, alpha=0.5, label='Normal CIFAR-10 Images', color='blue', density=True, edgecolor='black')

colors = ['red', 'orange', 'green', 'purple']
for (name, lls), color in zip(attack_lls.items(), colors):
    plt.hist(lls, bins=25, alpha=0.5, label=f'Attack: {name.upper()}', color=color, density=True, edgecolor='black')

# Zeichne den bisherigen Default-Threshold ein
plt.axvline(-100000, color='black', linestyle='--', label='Current Default Threshold (-100000)')

# Berechne automatische Vorschläge für einen echten Threshold
mean_normal = ll_normal.mean()
mean_maze = attack_lls.get("maze", np.array([0])).mean() if "maze" in attack_lls else 0
if mean_maze != 0:
    suggested_threshold = (mean_normal + mean_maze) / 2
    plt.axvline(suggested_threshold, color='red', linestyle='-.', linewidth=2, label=f'Suggested Threshold ({suggested_threshold:.0f})')

plt.title("Flow Matching Log-Likelihood Distributions (OOD Detection)", fontsize=16)
plt.xlabel("Log Likelihood (höher/rechts weicht meist von echten Verteilungen ab)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Mittelwerte der Log-Likelihoods:")
print(f"Normal (Baseline): {ll_normal.mean():.2f} ± {ll_normal.std():.2f}")
for name, lls in attack_lls.items():
    print(f"Attack - {name.upper()}: {lls.mean():.2f} ± {lls.std():.2f}")